### Pipeline Overview
- A complete, well‐documented Python pipeline designed for ingesting, cleaning, and merging heterogeneous datasets for predictive analytics.
- Built with careful consideration of the provided column information.

### Data Ingestion
- Pandas is used for handling CSV and Excel files.
- GeoPandas is employed for reading geodatabase layers.

### Customization and Adjustments
- File paths, skip‐row counts, and column names may need adjustment if your files differ from the provided samples.

### Data Merging
- Merging is performed on known keys:
    - Example: Using the realtor’s “zip_code” field.
    - Example: For crime data, a presumed “Location” field (modifiable as needed).
    - For mortgage and CPI data, the pipeline attaches the latest available value.

### For geospatial data (EPA Smart Location and Walkability):
- A placeholder merge is included.
- A proper merge typically requires a crosswalk (e.g., mapping zip codes to census tracts).

### Further Refinement
- The placeholder merge for geospatial data can be refined later using geocoding or an external crosswalk if available.

In [15]:
import os
import glob
import pandas as pd
import geopandas as gpd
import numpy as np
import json
import requests
from datetime import datetime
import matplotlib.pyplot as plt

# Basic Ingestion Functions for External Data (Mortgage, CPI, EPA, Walkability, Transit)
def load_excel_data(excel_path, sheet=0, skiprows=0):
    try:
        df = pd.read_excel(excel_path, sheet_name=sheet, skiprows=skiprows)
        print(f"Excel data loaded from: {excel_path} (Sheet: {sheet})")
        return df
    except Exception as e:
        print(f"Error loading Excel file {excel_path}: {e}")
        return pd.DataFrame()

def load_geodatabase_layer(gdb_path, layer_name):
    try:
        gdf = gpd.read_file(gdb_path, layer=layer_name)
        print(f"Geodatabase layer '{layer_name}' loaded from: {gdb_path}")
        return gdf
    except Exception as e:
        print(f"Error loading geodatabase layer '{layer_name}': {e}")
        return gpd.GeoDataFrame()

def load_transit_routes(csv_path):
    try:
        df = pd.read_csv(csv_path)
        print(f"Transit routes data loaded from: {csv_path}")
        if 'download_date' in df.columns:
            df['download_date'] = pd.to_datetime(df['download_date'], errors='coerce')
        return df
    except Exception as e:
        print(f"Error loading transit routes data: {e}")
        return pd.DataFrame()

# Helper Function to Load All Fatalities Files from a Folder
def get_fatalities_file_list(folder_path):
    """
    Scan the specified folder and return a list of file paths for all files
    ending with .csv and .csv.gz.
    """
    csv_files = glob.glob(os.path.join(folder_path, "*.csv"))
    csv_gz_files = glob.glob(os.path.join(folder_path, "*.csv.gz"))
    return csv_files + csv_gz_files

def load_storm_fatalities(file_paths):
    """
    Load and concatenate storm fatalities CSV/CSV.GZ files from a list of file paths.
    """
    df_list = []
    for fp in file_paths:
        try:
            if fp.endswith('.gz'):
                df = pd.read_csv(fp, compression="gzip")
            else:
                df = pd.read_csv(fp)
            print(f"Storm fatalities data loaded from: {fp} with {len(df)} rows.")
            if 'FATALITY_DATE' in df.columns:
                df['FATALITY_DATE'] = pd.to_datetime(df['FATALITY_DATE'], errors='coerce')
            df_list.append(df)
        except Exception as e:
            print(f"Error loading storm fatalities from {fp}: {e}")
    if df_list:
        combined_df = pd.concat(df_list, ignore_index=True)
        print(f"Combined storm fatalities dataset created with {len(combined_df)} rows.")
        return combined_df
    else:
        print("No storm fatalities files loaded; returning an empty DataFrame.")
        return pd.DataFrame()

# New Ingestion Functions for Zillow Data
def download_zillow_csv(url):
    """
    Download a Zillow CSV file from the given URL.
    """
    try:
        df = pd.read_csv(url)
        print(f"Data loaded from: {url}")
        return df
    except Exception as e:
        print(f"Error loading data from {url}: {e}")
        return pd.DataFrame()

def load_and_merge_zillow_data(urls):
    """
    Downloads the three Zillow datasets, filters out any rows where StateName == 'Puerto Rico',
    renames monthly columns to include a prefix (based on the metric), and merges them on
    common keys (RegionID, RegionName, StateName).
    """
    zillow_dfs = {}
    # Expected keys: "zhvi", "zori", "pending"
    for url in urls:
        df = download_zillow_csv(url)
        if df.empty:
            continue
        # Filter out rows where StateName equals "Puerto Rico" (case-insensitive)
        if 'StateName' in df.columns:
            df = df[df['StateName'].str.strip().str.upper() != "PUERTO RICO"]
        # Derive a metric key from the URL filename
        filename = url.split('/')[-1].split('?')[0]
        if "zhvi" in filename.lower():
            metric = "zhvi"
        elif "zori" in filename.lower():
            metric = "zori"
        elif "mean_doz_pending" in filename.lower() or "pending" in filename.lower():
            metric = "pending"
        else:
            metric = "zillow"
        # Rename monthly columns by prefixing with the metric (skip first five columns)
        fixed_cols = df.columns[:5]  # RegionID, SizeRank, RegionName, RegionType, StateName
        monthly_cols = df.columns[5:]
        df = df.rename(columns={col: f"{metric}_{col}" for col in monthly_cols})
        # Retain fixed columns unaltered
        df = df[list(fixed_cols) + [f"{metric}_{col}" for col in monthly_cols]]
        zillow_dfs[metric] = df

    # Merge the datasets on common keys: RegionID, RegionName, StateName.
    merged_df = None
    for metric, df in zillow_dfs.items():
        if merged_df is None:
            merged_df = df.copy()
        else:
            merged_df = pd.merge(merged_df, df, on=["RegionID", "RegionName", "StateName"], how="inner")
    if merged_df is not None:
        print("Zillow datasets merged successfully.")
        print("Merged Zillow DataFrame preview:")
        print(merged_df.head())
    else:
        print("No Zillow data available after merging.")
    return merged_df

# Preprocessing Functions (For External Data)
def preprocess_mortgage_data(df):
    if df.empty:
        return df
    df.columns = [str(col).strip().lower().replace(" ", "_") for col in df.columns]
    rate_col = None
    for col in df.columns:
        if "30" in col and "yr" in col:
            rate_col = col
            break
    if rate_col:
        df = df.rename(columns={rate_col: "mortgage_rate"})
        df["mortgage_rate"] = pd.to_numeric(df["mortgage_rate"], errors='coerce')
    else:
        print("Warning: 30-year mortgage rate column not found in mortgage data.")
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'], errors='coerce')
    return df

def preprocess_cpi_data(df):
    if df.empty:
        return df
    df = df.rename(columns={df.columns[0]: "cpi"})
    df["cpi"] = pd.to_numeric(df["cpi"], errors='coerce')
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
    return df

def preprocess_geospatial_data(gdf):
    if gdf.empty:
        return gdf
    if "GEOID10" in gdf.columns:
        gdf["GEOID10"] = gdf["GEOID10"].astype(str)
    return gdf

def preprocess_transit_routes(df):
    if df.empty:
        return df
    if 'download_date' in df.columns:
        df['download_date'] = pd.to_datetime(df['download_date'], errors='coerce')
    return df

def preprocess_storm_fatalities(df):
    if df.empty:
        return df
    df = df[df['FATALITY_TYPE'].notnull()]
    return df

# Merge External Data with Zillow Data

def merge_external_data(zillow_df, mortgage_df, cpi_df, epa_df, walkability_df, transit_df, fatalities_df, acs_df=None):
    """
    Attach aggregated external data to the base Zillow DataFrame.
    Since Zillow data is at the metro level (RegionID, RegionName, StateName),
    we attach aggregated or "latest" values from external sources as constant features.
    In a refined implementation, you would perform geographic joins.
    """
    merged_df = zillow_df.copy()
    
    # Attach latest mortgage rate (as a constant)
    if not mortgage_df.empty and "mortgage_rate" in mortgage_df.columns:
        latest_rate = mortgage_df["mortgage_rate"].dropna().iloc[-1]
        merged_df["mortgage_rate"] = latest_rate
    else:
        merged_df["mortgage_rate"] = np.nan
    
    # Attach latest CPI value
    if not cpi_df.empty and "cpi" in cpi_df.columns:
        latest_cpi = cpi_df["cpi"].dropna().iloc[-1]
        merged_df["cpi"] = latest_cpi
    else:
        merged_df["cpi"] = np.nan
    
    # Attach aggregated EPA SLC_score
    if not epa_df.empty and "GEOID10" in epa_df.columns and "SLC_score" in epa_df.columns:
        merged_df["epa_slc_score"] = epa_df["SLC_score"].mean()
    else:
        merged_df["epa_slc_score"] = np.nan
    
    # Attach aggregated Walkability Index
    if not walkability_df.empty and "NatWalkInd" in walkability_df.columns:
        merged_df["walkability_index"] = walkability_df["NatWalkInd"].mean()
    else:
        merged_df["walkability_index"] = np.nan
    
    # Attach aggregated Transit data (average route length)
    if not transit_df.empty and "Shape__Length" in transit_df.columns:
        merged_df["avg_route_length"] = transit_df["Shape__Length"].mean()
    else:
        merged_df["avg_route_length"] = np.nan
    
    # Attach Storm Fatalities: total number of fatalities (aggregated)
    if fatalities_df is not None and not fatalities_df.empty:
        total_fatalities = fatalities_df.shape[0]
        merged_df["total_fatalities"] = total_fatalities
    else:
        merged_df["total_fatalities"] = np.nan
    
    # (Optionally) Merge ACS data if available keyed by a common geographic variable
    if acs_df is not None and 'zip_code' in acs_df.columns:
        merged_df = merged_df.merge(acs_df, on='zip_code', how='left')
    
    return merged_df

# Extended Data Pipeline Assembly and Execution

def extended_data_pipeline():
    # Zillow dataset URLs
    zillow_urls = [
        "https://files.zillowstatic.com/research/public_csvs/zhvi/Metro_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv?t=1739388659",
        "https://files.zillowstatic.com/research/public_csvs/zori/Metro_zori_uc_sfrcondomfr_sm_month.csv?t=1739388659",
        "https://files.zillowstatic.com/research/public_csvs/mean_doz_pending/Metro_mean_doz_pending_uc_sfrcondo_sm_month.csv?t=1739388659"
    ]
    
    # External data file paths (adjust these to your environment)
    mortgage_path = "/Users/heonkim/Desktop/data/historicalweeklydata.xlsx"
    cpi_path = "/Users/heonkim/Desktop/data/SeriesReport-20250212132845_d75e74.xlsx"
    epa_gdb_path = "/Users/heonkim/Desktop/data/SmartLocationDatabaseV3/SmartLocationDatabase.gdb"
    walkability_gdb_path = "/Users/heonkim/Desktop/data/WalkabilityIndex/Natl_WI.gdb"
    transit_routes_path = "/Users/heonkim/Desktop/data/NTAD_National_Transit_Map_Routes_-6105509686316289189.csv"
    acs_json_path = "/Users/heonkim/Desktop/data/acs_5_years.json"  # ACS metadata (optional)
    fatalities_folder = "/Users/heonkim/Desktop/data/downloaded_csvs"  # Folder with CSV and CSV.GZ files
    
    # Ingest Zillow data and merge the three datasets
    zillow_df = load_and_merge_zillow_data(zillow_urls)
    if zillow_df is None or zillow_df.empty:
        print("Error: Zillow data could not be loaded or merged.")
        return pd.DataFrame()
    
    # Ingest external datasets
    mortgage_df = load_excel_data(mortgage_path, sheet=0, skiprows=3)
    cpi_df = load_excel_data(cpi_path, sheet=0, skiprows=3)
    epa_df = load_geodatabase_layer(epa_gdb_path, 'EPA_SLD_Database_V3')
    walkability_df = load_geodatabase_layer(walkability_gdb_path, 'NationalWalkabilityIndex')
    transit_df = load_transit_routes(transit_routes_path)
    # Optionally load ACS metadata (and query ACS data if desired)
    acs_metadata = load_acs_variables(acs_json_path)
    acs_df = None  # Placeholder for actual ACS data query
    
    # Load storm fatalities files by scanning the folder
    fatalities_files = glob.glob(os.path.join(fatalities_folder, "*.csv")) + \
                       glob.glob(os.path.join(fatalities_folder, "*.csv.gz"))
    fatalities_df = load_storm_fatalities(fatalities_files)
    
    # Preprocess external datasets as needed
    mortgage_df = preprocess_mortgage_data(mortgage_df)
    cpi_df = preprocess_cpi_data(cpi_df)
    epa_df = preprocess_geospatial_data(epa_df)
    walkability_df = preprocess_geospatial_data(walkability_df)
    transit_df = preprocess_transit_routes(transit_df)
    fatalities_df = preprocess_storm_fatalities(fatalities_df)
    
    # Merge external aggregated data with Zillow base data
    final_df = merge_external_data(zillow_df, mortgage_df, cpi_df, epa_df,
                                   walkability_df, transit_df, fatalities_df, acs_df)
    
    print("Final extended merged dataset preview:")
    print(final_df.head())
    return final_df

#Run the Extended Data Pipeline
if __name__ == "__main__":
    final_dataset = extended_data_pipeline()


Data loaded from: https://files.zillowstatic.com/research/public_csvs/zhvi/Metro_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv?t=1739388659
Data loaded from: https://files.zillowstatic.com/research/public_csvs/zori/Metro_zori_uc_sfrcondomfr_sm_month.csv?t=1739388659
Data loaded from: https://files.zillowstatic.com/research/public_csvs/mean_doz_pending/Metro_mean_doz_pending_uc_sfrcondo_sm_month.csv?t=1739388659
Zillow datasets merged successfully.
Merged Zillow DataFrame preview:
   RegionID  SizeRank_x       RegionName RegionType_x StateName  \
0    102001           0    United States      country       NaN   
1    394913           1     New York, NY          msa        NY   
2    753899           2  Los Angeles, CA          msa        CA   
3    394463           3      Chicago, IL          msa        IL   
4    394514           4       Dallas, TX          msa        TX   

   zhvi_2000-01-31  zhvi_2000-02-29  zhvi_2000-03-31  zhvi_2000-04-30  \
0    120362.209156    120573.964867  

/opt/anaconda3/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Geodatabase layer 'EPA_SLD_Database_V3' loaded from: /Users/heonkim/Desktop/data/SmartLocationDatabaseV3/SmartLocationDatabase.gdb
Geodatabase layer 'NationalWalkabilityIndex' loaded from: /Users/heonkim/Desktop/data/WalkabilityIndex/Natl_WI.gdb
Transit routes data loaded from: /Users/heonkim/Desktop/data/NTAD_National_Transit_Map_Routes_-6105509686316289189.csv
ACS metadata loaded from: /Users/heonkim/Desktop/data/acs_5_years.json
Storm fatalities data loaded from: /Users/heonkim/Desktop/data/downloaded_csvs/ugc_areas.csv with 7790 rows.
Storm fatalities data loaded from: /Users/heonkim/Desktop/data/downloaded_csvs/StormEvents_fatalities-ftp_v1.0_d1983_c20220425.csv.gz with 30 rows.
Storm fatalities data loaded from: /Users/heonkim/Desktop/data/downloaded_csvs/StormEvents_fatalities-ftp_v1.0_d1963_c20210803.csv.gz with 17 rows.
Storm fatalities data loaded from: /Users/heonkim/Desktop/data/downloaded_csvs/StormEvents_details-ftp_v1.0_d2023_c20241216.csv.gz with 75596 rows.
Storm fatal

/var/folders/vr/txw5f_kj4h9653_2bzcpzn6w0000gn/T/ipykernel_10354/2991283128.py:65: DtypeWarning: Columns (28) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(fp, compression="gzip")


Storm fatalities data loaded from: /Users/heonkim/Desktop/data/downloaded_csvs/StormEvents_details-ftp_v1.0_d1998_c20220425.csv.gz with 50973 rows.
Storm fatalities data loaded from: /Users/heonkim/Desktop/data/downloaded_csvs/StormEvents_fatalities-ftp_v1.0_d1993_c20220425.csv.gz with 27 rows.
Storm fatalities data loaded from: /Users/heonkim/Desktop/data/downloaded_csvs/StormEvents_fatalities-ftp_v1.0_d1975_c20220425.csv.gz with 30 rows.
Storm fatalities data loaded from: /Users/heonkim/Desktop/data/downloaded_csvs/StormEvents_fatalities-ftp_v1.0_d2002_c20220425.csv.gz with 466 rows.
Storm fatalities data loaded from: /Users/heonkim/Desktop/data/downloaded_csvs/StormEvents_fatalities-ftp_v1.0_d2021_c20240716.csv.gz with 1266 rows.
Storm fatalities data loaded from: /Users/heonkim/Desktop/data/downloaded_csvs/StormEvents_locations-ftp_v1.0_d1980_c20220425.csv.gz with 0 rows.
Storm fatalities data loaded from: /Users/heonkim/Desktop/data/downloaded_csvs/StormEvents_fatalities-ftp_v1.0_

...

Storm fatalities data loaded from: /Users/heonkim/Desktop/data/downloaded_csvs/StormEvents_details-ftp_v1.0_d1996_c20220425.csv.gz with 48561 rows.
Storm fatalities data loaded from: /Users/heonkim/Desktop/data/downloaded_csvs/StormEvents_fatalities-ftp_v1.0_d2019_c20240117.csv.gz with 733 rows.
Storm fatalities data loaded from: /Users/heonkim/Desktop/data/downloaded_csvs/StormEvents_locations-ftp_v1.0_d1986_c20220425.csv.gz with 0 rows.
Storm fatalities data loaded from: /Users/heonkim/Desktop/data/downloaded_csvs/StormEvents_details-ftp_v1.0_d2024_c20250122.csv.gz with 64332 rows.
Storm fatalities data loaded from: /Users/heonkim/Desktop/data/downloaded_csvs/StormEvents_locations-ftp_v1.0_d2006_c20250122.csv.gz with 136174 rows.
Storm fatalities data loaded from: /Users/heonkim/Desktop/data/downloaded_csvs/StormEvents_fatalities-ftp_v1.0_d2004_c20220425.csv.gz with 369 rows.
Storm fatalities data loaded from: /Users/heonkim/Desktop/data/downloaded_csvs/StormEvents_fatalities-ftp_v1.

/var/folders/vr/txw5f_kj4h9653_2bzcpzn6w0000gn/T/ipykernel_10354/2991283128.py:65: DtypeWarning: Columns (26,48) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(fp, compression="gzip")


Storm fatalities data loaded from: /Users/heonkim/Desktop/data/downloaded_csvs/StormEvents_details-ftp_v1.0_d2010_c20220425.csv.gz with 62807 rows.
Storm fatalities data loaded from: /Users/heonkim/Desktop/data/downloaded_csvs/StormEvents_locations-ftp_v1.0_d1982_c20220425.csv.gz with 0 rows.
Storm fatalities data loaded from: /Users/heonkim/Desktop/data/downloaded_csvs/StormEvents_details-ftp_v1.0_d1978_c20220425.csv.gz with 3657 rows.
Storm fatalities data loaded from: /Users/heonkim/Desktop/data/downloaded_csvs/StormEvents_fatalities-ftp_v1.0_d1971_c20210803.csv.gz with 31 rows.
Storm fatalities data loaded from: /Users/heonkim/Desktop/data/downloaded_csvs/StormEvents_fatalities-ftp_v1.0_d1977_c20220425.csv.gz with 15 rows.
Storm fatalities data loaded from: /Users/heonkim/Desktop/data/downloaded_csvs/StormEvents_fatalities-ftp_v1.0_d1991_c20220425.csv.gz with 42 rows.
Storm fatalities data loaded from: /Users/heonkim/Desktop/data/downloaded_csvs/StormEvents_fatalities-ftp_v1.0_d200

/var/folders/vr/txw5f_kj4h9653_2bzcpzn6w0000gn/T/ipykernel_10354/2991283128.py:75: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_df = pd.concat(df_list, ignore_index=True)


Combined storm fatalities dataset created with 3691629 rows.
Final extended merged dataset preview:
   RegionID  SizeRank_x       RegionName RegionType_x StateName  \
0    102001           0    United States      country       NaN   
1    394913           1     New York, NY          msa        NY   
2    753899           2  Los Angeles, CA          msa        CA   
3    394463           3      Chicago, IL          msa        IL   
4    394514           4       Dallas, TX          msa        TX   

   zhvi_2000-01-31  zhvi_2000-02-29  zhvi_2000-03-31  zhvi_2000-04-30  \
0    120362.209156    120573.964867    120836.573128    121399.816300   
1    215827.415564    216744.560306    217670.199462    219545.902568   
2    218392.422126    219205.119030    220287.623555    222441.443922   
3    150388.567381    150527.783385    150792.922716    151452.523481   
4    125052.500914    125108.199735    125172.231232    125338.514302   

   zhvi_2000-05-31  ...  pending_2024-09-30  pending_2024-

In [16]:
final_dataset.to_csv("final_dataset.csv", index=False)

### Data Preprocessing

In [6]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Functions to Summarize Monthly Time Series Data for a Given Metric

def summarize_time_series(df, metric_prefix):
    """
    For a given metric (e.g., 'zhvi'), identify all monthly columns (assumed to be named like 'zhvi_YYYY-MM-DD'),
    sort them chronologically, and compute summary features for each row:
      - Mean
      - Standard deviation (volatility)
      - Trend slope (using linear regression on the time index)
      - Last observed value
      - Growth rate over the period (percentage change from first to last)
    
    The computed features are added as new columns to the DataFrame.
    """
    # Identify columns starting with the metric prefix + "_"
    ts_cols = [col for col in df.columns if col.startswith(metric_prefix + "_")]
    if not ts_cols:
        print(f"No columns found for metric '{metric_prefix}'")
        return df

    # Sort columns by date extracted from column names
    try:
        ts_cols_sorted = sorted(ts_cols, key=lambda x: datetime.strptime(x.replace(metric_prefix + "_", ""), "%Y-%m-%d"))
    except Exception as e:
        print(f"Error sorting columns for {metric_prefix}: {e}")
        ts_cols_sorted = ts_cols

    # Convert date parts to numeric (using ordinal)
    dates = [datetime.strptime(col.replace(metric_prefix + "_", ""), "%Y-%m-%d") for col in ts_cols_sorted]
    time_numeric = np.array([d.toordinal() for d in dates])

    means, stds, slopes, last_vals, growth_rates = [], [], [], [], []
    
    for idx, row in df.iterrows():
        try:
            values = row[ts_cols_sorted].astype(float).values
        except Exception as e:
            values = np.array([np.nan]*len(ts_cols_sorted))
        if np.all(np.isnan(values)):
            means.append(np.nan)
            stds.append(np.nan)
            slopes.append(np.nan)
            last_vals.append(np.nan)
            growth_rates.append(np.nan)
        else:
            mean_val = np.nanmean(values)
            std_val = np.nanstd(values)
            valid_mask = ~np.isnan(values)
            if valid_mask.sum() > 1:
                slope, intercept = np.polyfit(time_numeric[valid_mask], values[valid_mask], 1)
            else:
                slope = np.nan
            last_val = values[-1] if not np.isnan(values[-1]) else np.nan
            first_val = values[0] if not np.isnan(values[0]) else np.nan
            growth_rate = ((last_val - first_val) / first_val) if first_val and first_val != 0 else np.nan

            means.append(mean_val)
            stds.append(std_val)
            slopes.append(slope)
            last_vals.append(last_val)
            growth_rates.append(growth_rate)
    
    # Add new summary feature columns to the DataFrame
    df[f"{metric_prefix}_mean"] = means
    df[f"{metric_prefix}_std"] = stds
    df[f"{metric_prefix}_slope"] = slopes
    df[f"{metric_prefix}_last"] = last_vals
    df[f"{metric_prefix}_growth"] = growth_rates
    
    return df

def process_time_series_features(df, metric_prefixes):
    """
    Apply the summarize_time_series function for each metric prefix.
    """
    for prefix in metric_prefixes:
        df = summarize_time_series(df, prefix)
    return df

# Functions to Clean, Impute, Scale, and Remove Redundancy

def clean_impute_scale(df):
    """
    - Ensure column names are unique and stripped of whitespace.
    - Drop redundant duplicate columns (e.g., those ending with '_x' or '_y').
    - Drop numeric columns that are entirely missing.
    - Impute missing numeric values using the median.
    - Standardize numeric variables using StandardScaler.
    """
    # Strip whitespace and remove duplicate column names
    df.columns = df.columns.str.strip()
    df = df.loc[:, ~df.columns.duplicated()]
    
    # Identify numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    print("Original numeric columns:", numeric_cols)
    
    # Extract numeric data and drop columns that are entirely missing
    numeric_data = df[numeric_cols].copy()
    numeric_data = numeric_data.dropna(axis=1, how='all')
    updated_numeric_cols = numeric_data.columns.tolist()
    print("Numeric columns after dropping all-missing columns:", updated_numeric_cols)
    
    # Impute missing values with median
    imputer = SimpleImputer(strategy='median')
    imputed_values = imputer.fit_transform(numeric_data)
    imputed_df = pd.DataFrame(imputed_values, columns=updated_numeric_cols, index=df.index)
    df[updated_numeric_cols] = imputed_df

    # Scale numeric columns
    scaler = StandardScaler()
    scaled_values = scaler.fit_transform(df[updated_numeric_cols])
    scaled_df = pd.DataFrame(scaled_values, columns=updated_numeric_cols, index=df.index)
    df[updated_numeric_cols] = scaled_df
    
    return df

# Function to Define a Target Variable

def define_target(df, metric_prefix="zhvi"):
    """
    Define a binary target variable for risk assessment.
    Here, risk is defined as 1 if the computed 'zhvi_growth' is negative,
    and 0 otherwise.
    """
    target_col = f"{metric_prefix}_growth"
    if target_col in df.columns:
        df['risk'] = df[target_col].apply(lambda x: 1 if pd.notnull(x) and x < 0 else 0)
    else:
        print(f"Target column {target_col} not found. Cannot define target variable.")
        df['risk'] = np.nan
    return df

# Main Function to Prepare the Prediction-Ready Dataset

def prepare_prediction_dataset(file_path="final_dataset.csv"):
    """
    Load the final merged dataset, process time-series features, clean/impute/scale,
    eliminate redundant columns, and define a target variable.
    
    The processed dataset is saved to 'prediction_ready_dataset.csv'.
    """
    # Load the merged dataset
    df = pd.read_csv(file_path)
    print("Loaded merged dataset with shape:", df.shape)
    
    # Process time series features for each metric (assumed to be prefixed with "zhvi_", "zori_", "pending_")
    metric_prefixes = ["zhvi", "zori", "pending"]
    df = process_time_series_features(df, metric_prefixes)
    print("Time series features computed. New shape:", df.shape)
    
    # Clean, impute missing values, scale numeric features, and remove redundant columns
    df = clean_impute_scale(df)
    print("Missing values imputed and numeric features scaled. New shape:", df.shape)
    
    # Define the target variable based on zhvi_growth (for risk assessment)
    df = define_target(df, metric_prefix="zhvi")
    print("Target variable 'risk' defined based on zhvi_growth.")
    
    # Save the processed dataset to a new CSV file
    output_file = "prediction_ready_dataset.csv"
    df.to_csv(output_file, index=False)
    print(f"Prediction-ready dataset saved to {output_file}")
    
    return df

# Execution

if __name__ == "__main__":
    # Replace "final_dataset.csv" with the correct path to your merged dataset file.
    prediction_df = prepare_prediction_dataset("final_dataset.csv")
    print("Preview of the prediction-ready dataset:")
    print(prediction_df.head())


Loaded merged dataset with shape: (518, 517)
Time series features computed. New shape: (518, 532)
Original numeric columns: ['RegionID', 'SizeRank_x', 'zhvi_2000-01-31', 'zhvi_2000-02-29', 'zhvi_2000-03-31', 'zhvi_2000-04-30', 'zhvi_2000-05-31', 'zhvi_2000-06-30', 'zhvi_2000-07-31', 'zhvi_2000-08-31', 'zhvi_2000-09-30', 'zhvi_2000-10-31', 'zhvi_2000-11-30', 'zhvi_2000-12-31', 'zhvi_2001-01-31', 'zhvi_2001-02-28', 'zhvi_2001-03-31', 'zhvi_2001-04-30', 'zhvi_2001-05-31', 'zhvi_2001-06-30', 'zhvi_2001-07-31', 'zhvi_2001-08-31', 'zhvi_2001-09-30', 'zhvi_2001-10-31', 'zhvi_2001-11-30', 'zhvi_2001-12-31', 'zhvi_2002-01-31', 'zhvi_2002-02-28', 'zhvi_2002-03-31', 'zhvi_2002-04-30', 'zhvi_2002-05-31', 'zhvi_2002-06-30', 'zhvi_2002-07-31', 'zhvi_2002-08-31', 'zhvi_2002-09-30', 'zhvi_2002-10-31', 'zhvi_2002-11-30', 'zhvi_2002-12-31', 'zhvi_2003-01-31', 'zhvi_2003-02-28', 'zhvi_2003-03-31', 'zhvi_2003-04-30', 'zhvi_2003-05-31', 'zhvi_2003-06-30', 'zhvi_2003-07-31', 'zhvi_2003-08-31', 'zhvi_2003-09

/var/folders/vr/txw5f_kj4h9653_2bzcpzn6w0000gn/T/ipykernel_10793/579628840.py:140: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['risk'] = df[target_col].apply(lambda x: 1 if pd.notnull(x) and x < 0 else 0)


Next steps: 

1. Loads prediction-ready dataset (saved as "prediction_ready_dataset.csv").
2. Selects only the numeric features (since most ML algorithms require numeric input).
3. Splits the data into training and testing sets.
4. Trains both a Logistic Regression and an SVM classifier.
5. Evaluates their performance using accuracy, confusion matrix, and a classification report.